In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, UTC
import uuid   

In [0]:
%sql
create schema if not exists e_com_adb.silver_schema;

In [0]:
silver_run_id = str(uuid.uuid4())
print(silver_run_id)

In [0]:
%sql
create table if not exists e_com_adb.silver_schema.processing_control(
    layer string,
    entity_name string,
    last_processed_bronze_run_id string,
    last_processed_bronze_ingested_at timestamp,
    rows_merged bigint,
    run_status string,
    silver_run_id string,
    updated_at timestamp
) using delta;

In [0]:
def upsert_to_silver(df_source, target_table, join_key):
    if (spark.catalog.tableExists(target_table)):
        dt = DeltaTable.forName(spark, target_table).alias("t")

        (
            dt.merge(
                df_source.alias('s'),
                f't.{join_key} = s.{join_key}'
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        df_source.write.format('delta').saveAsTable(target_table)


In [0]:
def get_last_processed_bronze_ingested_at(entity_name: str):
    ctrl = (
        spark
        .table('e_com_adb.silver_schema.processing_control')
        .filter(
            (col('layer') == 'silver') &
            (col('entity_name') == entity_name) &
            (col('run_status') == 'SUCCESS')
        )
        .orderBy(col('updated_at').desc())
        .limit(1)
    )

    rows = ctrl.collect()

    if not rows:
        return None
    else:
        return rows[0]['last_processed_bronze_ingested_at']
    

In [0]:
def upsert_silver_control(entity_name, last_processed_bronze_run_id, last_processed_bronze_ingested_at, rows_merged):
    ctrl_df = spark.createDataFrame([(
        "silver",
        entity_name,
        last_processed_bronze_run_id,
        last_processed_bronze_ingested_at,
        int(rows_merged),
        "SUCCESS",
        silver_run_id,
        datetime.now(UTC)
    )], 
        schema="""
            layer STRING,
            entity_name STRING,
            last_processed_bronze_run_id STRING,
            last_processed_bronze_ingested_at TIMESTAMP,
            rows_merged BIGINT,
            run_status STRING,
            silver_run_id STRING,
            updated_at TIMESTAMP
        """
    )

    dt = DeltaTable.forName(spark, 'e_com_adb.silver_schema.processing_control').alias('t')

    (
        dt.merge(
            ctrl_df.alias('s'),
            f't.entity_name = s.entity_name and t.layer = s.layer'
        )
        .whenMatchedUpdate(
            set = {
                'last_processed_bronze_run_id': 's.last_processed_bronze_run_id',
                'last_processed_bronze_ingested_at': 's.last_processed_bronze_ingested_at',
                'rows_merged': 's.rows_merged',
                'run_status': 's.run_status',
                'silver_run_id': 's.silver_run_id',
                'updated_at': 's.updated_at',                
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
def get_incremental_bronze(bronze_table, entity_name):
    last_ingested_at = get_last_processed_bronze_ingested_at(entity_name)
    bronze_df = spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df, last_ingested_at

    return bronze_df.filter(col('bronze_ingested_at') > 'last_ingested_at'), last_ingested_at

In [0]:
df_raw = spark.read.table('e_com_adb.bronze_schema.orders_raw')
display(df_raw)

In [0]:
orders_inc, last_orders_ingested_at = get_incremental_bronze(
    "e_com_adb.bronze_schema.orders_raw",
    "orders"
)

orders_inc_count = orders_inc.count()
print(f'orders row_to_process_in_silver = {orders_inc_count}')

if orders_inc_count > 0:
    order_window = (
        Window
        .partitionBy('order_id')
        .orderBy(
            to_timestamp(col('updated_at')).desc(), 
            col('bronze_ingested_at').desc()
        )        
    )

    orders_cleaned = (
        orders_inc
        .withColumns({
            'order_status': upper(trim(col('order_status'))),
            'order_amount': regexp_replace(col('order_amount'), r"[$, ]", ""),
        })
        .withColumns(
            {                
                'order_status': when(col('order_status') == '', lit(Nones)).otherwise(col('order_status')),
                'order_amount': when(trim(col('order_amount')).isin('N/A', "NULL", "??", ""), lit(None)).otherwise(col('order_amount').cast(DoubleType())),
                'created_at': to_timestamp(col('created_at')),
                'updated_at': to_timestamp(col('updated_at')),
                'row_rank': row_number().over(order_window),
                'silver_run_id': lit(silver_run_id)
            }
        )
        .where(col('row_rank') == 1)
        .drop("row_rank")
    )

    upsert_to_silver(
        orders_cleaned, 
        'e_com_adb.silver_schema.orders_cleaned', 
        'order_id'
    )

    orders_validated = (
        orders_cleaned
        .withColumns({
            "to_be_verified_by_orders_team": (
                when(col('customer_id').isNull(), "verify_customer_id")
                .when(col('product_id').isNull(), "verify_product_id")
                .when((col('order_status').isNull()) | (trim(col('order_status')) == ''), "verify_order_status")
                .when((col('order_amount').isNull()) | (col('order_amount') <= 0), "verify_order_amount")
                .otherwise(lit('No Issues'))            
            ),
            "check_order_amount": (
                when((col('order_amount').isNull()) | (col('order_amount') <= 0), lit(True))
                .otherwise(lit(False))
            ),
            "order_date": to_date(col('created_at')),            
        })
        .withColumns({
            "order_year": year(col('created_at')),
            "order_month": month(col('created_at')),
            "order_day": dayofmonth(col('created_at')),
            "order_dow": date_format(col('created_at'), "E"),
        })
    )

    orders_good = (
        orders_validated
        .where(col('to_be_verified_by_orders_team') == 'No Issues')
    )
    
    orders_bad = (
        orders_validated
        .where(col('to_be_verified_by_orders_team') != 'No Issues')
        .withColumn('quarantine_ts', current_timestamp())
    )

    upsert_to_silver(
        orders_good,
        'e_com_adb.silver_schema.orders_transformed',
        'order_id'
    )

    orders_bad.write.format('delta').mode('append').saveAsTable('e_com_adb.silver_schema.orders_quarantine')

    mx_ingested = orders_inc.agg(max(col('bronze_ingested_at')).alias('mx')).collect()[0]['mx']
    mx_run = (
        orders_inc
        .filter(col('bronze_ingested_at') == lit(mx_ingested))
        .agg(max(col('bronze_run_id')).alias('mx'))
        .collect()[0]['mx']
    )

    upsert_silver_control('orders', mx_run, mx_ingested, orders_good.count())
    
else:
    print('No new orders bronze rows for silver')
    upsert_silver_control('orders', None, last_orders_ingested_at, orders_inc_count)

In [0]:
display(spark.table('e_com_adb.silver_schema.orders_cleaned'))

In [0]:
# Step 5 - Products incremental processing
# Read only the Bronze product rows that Silver has not processed yet.
products_inc, last_products_ingested_at = get_incremental_bronze("e_com_adb.bronze_schema.products_raw", "products")

# Count the incremental product rows entering Silver in this run.
products_inc_count = products_inc.count()
print(f"products rows_to_process_in_silver = {products_inc_count}")

if products_inc.count() > 0:
    # Create a window that keeps the latest product record for each product_id.
    product_window = Window.partitionBy("product_id").orderBy(
        col("updated_at").cast("timestamp").desc(),
        col("bronze_ingested_at").desc()
    )
    

    # Start the Silver product-cleaning pipeline. This block standardizes and deduplicates raw product records.
    products_cleaned = (
        products_inc
        # Standardize product_name by trimming spaces and converting text to uppercase.
        .withColumn("product_name", upper(trim(col("product_name"))))
        .withColumn("product_name", when(col("product_name") == "", lit(None)).otherwise(col("product_name")))
        .withColumn(
        "category",
        when(upper(trim(col("category"))).contains("ELECTRNICS"), "ELECTRONICS")
        # .when(upper(trim(col("category"))) == "FITNESS", "FITNESS")
        # .when(upper(trim(col("category"))) == "LIFESTYLE", "LIFESTYLE")
        .otherwise(upper(trim(col("category"))))
        )
    
        # Start cleaning the product price field before converting it to numeric.
        .withColumn("price", trim(col("price")))
        .withColumn("price", regexp_replace(col("price"), r"\$", ""))
        .withColumn("price", regexp_replace(col("price"), ",","."))
        .withColumn("price", regexp_replace(col("price"), r"\s+", ""))
        .withColumn("price", col('price').try_cast(DoubleType()))
        .withColumn("updated_at", to_timestamp("updated_at"))
        # Assign a row number inside each business key so we can keep only the latest version of that record.
        .withColumn("row_rank", row_number().over(product_window))
        # Keep only the latest record for each business key.
        .filter(col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", lit(silver_run_id))
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(products_cleaned, "e_com_adb.silver_schema.products_cleaned", "product_id")

    # Apply Silver data-quality rules to the cleaned product records.
    products_validated = (
        products_cleaned
        .withColumn(
            "to_be_verified_by_products_team",
            when(col("product_name").isNull(), "verify_product_name")
            .when(col("category").isNull(), "verify_category")
            .when(col("price").isNull() |(col("price") <= 0), "verify_price")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_product_price",
            when(col("price").isNull()|(col("price") <= 0), "invalid_price").otherwise("valid_price")
        )        
    )

    # Keep only valid product rows for the transformed Silver table.
    products_good = products_validated.filter(
        (col("to_be_verified_by_products_team") == "No Issues") &
        (col("check_product_price") == "valid_price")
    )

    if "price_raw" in products_good.columns:
        # Keep only valid product rows for the transformed Silver table.
        products_good = products_good.drop("price_raw")

    # Send invalid product rows to the quarantine dataset for manual review.
    products_bad = products_validated.filter(
        (col("to_be_verified_by_products_team") != "No Issues") |
        (col("check_product_price") == "invalid_price")
    ).withColumn("quarantine_ts", current_timestamp())

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(products_good, "e_com_adb.silver_schema.products_transformed", "product_id")
    # Append bad product rows to the quarantine table instead of losing them.
    products_bad.write.format("delta").mode("append").saveAsTable("e_com_adb.silver_schema.products_quarantine")

    mx_ingested = products_inc.agg(max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = products_inc.filter(col("bronze_ingested_at") == lit(mx_ingested)).agg(max("bronze_run_id").alias("mx")).collect()[0]["mx"]
    
    upsert_silver_control("products", mx_run, mx_ingested, products_good.count())
else:
    print("No new products Bronze rows for Silver.")
    upsert_silver_control(
        "products",
        None,
        last_products_ingested_at,
        products_inc_count
    )

In [0]:
payments_inc, last_payments_ingested_at = get_incremental_bronze("e_com_adb.bronze_schema.payments_raw", "payments")
print("Payments last processed Bronze ingested_at =", last_payments_ingested_at)

# Count the incremental payment rows entering Silver in this run.
payments_inc_count = payments_inc.count()
print(f"payments rows_to_process_in_silver = {payments_inc_count}")

if payments_inc_count > 0:
    payment_window = Window.partitionBy("payment_id").orderBy(
        col("processed_at").cast("timestamp").desc(),
        col("bronze_ingested_at").desc()
    )
    # Start the Silver payment-cleaning pipeline. This block standardizes and deduplicates raw payment records.
    payments_cleaned = (
        payments_inc
        .withColumn("payment_status", upper(trim(col("payment_status"))))
        .withColumn("payment_status", when(col("payment_status") == "", lit(None)).otherwise(col("payment_status")))
        # Start cleaning the payment amount field before converting it to numeric.
        .withColumn("paid_amount", trim(col("paid_amount")))
        .withColumn("paid_amount", regexp_replace(col("paid_amount"), r"\$", ""))
        .withColumn("paid_amount", regexp_replace(col("paid_amount"), ",", "."))
        .withColumn("paid_amount", regexp_replace(col("paid_amount"), r"\s+", ""))
        .withColumn("paid_amount", expr("try_cast(paid_amount as double)"))
        .withColumn("processed_at", to_timestamp("processed_at"))
        # Assign a row number inside each business key so we can keep only the latest version of that record.
        .withColumn("row_rank", row_number().over(payment_window))
        # Keep only the latest record for each business key.
        .filter(col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", lit(silver_run_id))
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(payments_cleaned, "e_com_adb. silver_schema.payments_cleaned", "payment_id")

    # Apply Silver data-quality rules to the cleaned payment records.
    payments_validated = (
        payments_cleaned
        .withColumn(
            "to_be_verified_by_payments_team",
            when(col("order_id").isNull(), "verify_order_id")
            .when(col("payment_status").isNull(), "verify_payment_status")
            .when(col("paid_amount").isNull() | (col("paid_amount") <= 0), "verify_paid_amount")
            .otherwise("No Issues")
        )
        .withColumn("check_paid_amoint",
            when(col("paid_amount").isNull() |(col("paid_amount") <= 0), lit(True))
            .otherwise(lit(False))
        )
    )
    

    # Keep only valid payment rows for the transformed Silver table.
    payments_good = payments_validated.filter(col("to_be_verified_by_payments_team") == "No Issues")
    # Send invalid payment rows to the quarantine dataset for manual review.
    payments_bad = payments_validated. filter(col("to_be_verified_by_payments_team") != "No Issues").withColumn("quarantine_ts", 
    current_timestamp())

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(payments_good, "e_com_adb.silver_schema.payments_transformed", "payment_id")
    # Append bad payment rows to the quarantine table instead of losing them.
    payments_bad.write.format("delta").mode("append").saveAsTable("e_com_adb.silver_schema.payments_quarantine")

    mx_ingested = payments_inc.agg(max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = payments_inc.filter(col("bronze_ingested_at") == lit(mx_ingested)).agg(max("bronze_run_id").alias("mx")).collect()[0]["mx"]
    upsert_silver_control("payments", mx_run, mx_ingested, payments_good.count())
else:
    print("No new payments Bronze rows for Silver.")
    upsert_silver_control(
        "payments",
        None,
        last_payments_ingested_at,
        payments_inc_count
    )

In [0]:
cleaned_df = spark.table("e_com_adb.silver_schema.payments_cleaned")
quarantine_df = spark.table("e_com_adb.silver_schema.payments_quarantine")
transformed_df = spark.table("e_com_adb.silver_schema.payments_transformed")

print(f'Cleaned count: {cleaned_df.count()}')
print(f'Transformed count: {transformed_df.count()}')
print(f'Quarantine count: {quarantine_df.count()}\n')

display(spark.table("e_com_adb.silver_schema.processing_control"))